# LLMバッチ推論による名寄せパイプラインの構築

- 作成日: 2025-05-18
- 実行環境: Serverless SQL Warehouse

## 準備

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW item_master AS
SELECT * FROM 
read_files(
  '/Volumes/skato/entity_matching/raw/item_master.csv', -- カタログ名skato、スキーマ名entity_matching、ボリューム名rawに配置したitem_master.csvを読み込み
  format => 'csv',
  header => true,
  mode => 'FAILFAST'
);
SELECT * FROM item_master;

jan_code,name,_rescued_data
4979874466846,組ねじ 4.5×10 ［20本 (4本×5セット)］ #ネジ 八幡ねじ YAHATA DIY 通販,null
4979874466846,八幡ねじ 組ねじ 4.5×10mm,null
4979874466846,YAHATA 組ねじ,null
4979874453792,八幡ねじ ステントラスタッピング 3×12mm,null
4979874453792,ステントラスタッピング 3×12 1セット(14個×5パック入) 取り寄せ商品,null
4979874453792,ステンレス十字穴付きトラスタッピング 3×12mm,null
4979874220462,八幡ねじ 中空用アンカー 4S よーと,null
4979874220462,中空用アンカー ﾁｭｳｸｳﾖｳｱﾝｶｰ | 八幡ねじ,null
4979874220462,中空用アンカー(4コ入) ヨート4S,null
4903320162143,S字フック ステンレス 大 6個入,null


## 実装

### パターン1. 対象を一意に特定するIDを持つ

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW new_name_list_with_jan_code AS
SELECT 
  jan_code
  , cast(collect_list(name) as STRING) as names
  , ai_query( -- ①new_name: JANコード毎に品目名を名寄せ
      'databricks-meta-llama-3-1-405b-instruct',
      CONCAT('提供されたリストは表記揺れしているが同じJANコードを持つ品目です。これを1つの品目名に名寄せしてください。名寄せ結果以外の文章は返却しないでください。テキスト: ', names)
    ) AS new_name
  , ai_query( -- ②new_name_classified: JANコード毎に品目名を名寄せし、かつ製造者やスペック情報を分類する
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('提供されたリストは表記揺れしているが同じJANコードを持つ品目です。これを1つの品目、製造者、スペックに分解・名寄せしてください。名寄せ結果以外の文章は返却しないでください。テキスト: ', names),
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "name_collection",
        "schema": {
          "type": "object",
          "properties": {
            "new_name": {"type": "string"},
            "manufacturer": {"type": "string"},
            "spec": {"type": "string"}
          }
        }
      }
    }') AS new_name_classified
FROM 
  item_master
GROUP BY
  1
;
SELECT * FROM new_name_list_with_jan_code;

jan_code,names,new_name,new_name_classified
4979874466846,"[組ねじ 4.5×10 ［20本 (4本×5セット)］ #ネジ 八幡ねじ YAHATA DIY 通販, 八幡ねじ 組ねじ 4.5×10mm, YAHATA 組ねじ]",八幡ねじ 組ねじ 4.5×10mm,"{""new_name"": ""組ねじ"", ""manufacturer"": ""八幡ねじ"", ""spec"": ""4.5×10mm""}"
4903320162143,"[S字フック ステンレス 大 6個入, レック S字フックステンレス（大） H00505 6個入, レック S字フックステンレス（大） 耐荷重9kg H00505 1パック（6個）]",レック S字フックステンレス（大）6個入,"{""new_name"": ""S字フック ステンレス 大 6個入"", ""manufacturer"": ""レック"", ""spec"": ""耐荷重9kg""}"
4979874453792,"[八幡ねじ ステントラスタッピング 3×12mm, ステントラスタッピング 3×12 1セット(14個×5パック入) 取り寄せ商品, ステンレス十字穴付きトラスタッピング 3×12mm]",ステントラスタッピング 3×12mm,"{""new_name"": ""ステントラスタッピング"", ""manufacturer"": ""八幡ねじ"", ""spec"": ""3×12mm""}"
4979874220462,"[八幡ねじ 中空用アンカー 4S よーと, 中空用アンカー ﾁｭｳｸｳﾖｳｱﾝｶｰ | 八幡ねじ, 中空用アンカー(4コ入) ヨート4S]",八幡ねじ 中空用アンカー 4S,"{""new_name"": ""中空用アンカー"", ""manufacturer"": ""八幡ねじ"", ""spec"": ""4S""}"


In [0]:
%sql
SELECT
  t1.jan_code
  , t1.name
  , t2.new_name
  , t2.new_name_classified:new_name
  , t2.new_name_classified:manufacturer
  , t2.new_name_classified:spec
FROM
  item_master t1
  LEFT OUTER JOIN new_name_list_with_jan_code t2 ON t1.jan_code = t2.jan_code
;

jan_code,name,new_name,new_name,manufacturer,spec
4979874466846,組ねじ 4.5×10 ［20本 (4本×5セット)］ #ネジ 八幡ねじ YAHATA DIY 通販,八幡ねじ 組ねじ 4.5×10mm,組ねじ,八幡ねじ,4.5×10
4979874466846,八幡ねじ 組ねじ 4.5×10mm,八幡ねじ 組ねじ 4.5×10mm,組ねじ,八幡ねじ,4.5×10
4979874466846,YAHATA 組ねじ,八幡ねじ 組ねじ 4.5×10mm,組ねじ,八幡ねじ,4.5×10
4903320162143,S字フック ステンレス 大 6個入,レック S字フックステンレス（大）6個入,S字フック ステンレス 大 6個入,レック,耐荷重9kg
4903320162143,レック S字フックステンレス（大） H00505 6個入,レック S字フックステンレス（大）6個入,S字フック ステンレス 大 6個入,レック,耐荷重9kg
4903320162143,レック S字フックステンレス（大） 耐荷重9kg H00505 1パック（6個）,レック S字フックステンレス（大）6個入,S字フック ステンレス 大 6個入,レック,耐荷重9kg
4979874453792,八幡ねじ ステントラスタッピング 3×12mm,ステントラスタッピング 3×12mm,ステントラスタッピング,八幡ねじ,3×12mm
4979874453792,ステントラスタッピング 3×12 1セット(14個×5パック入) 取り寄せ商品,ステントラスタッピング 3×12mm,ステントラスタッピング,八幡ねじ,3×12mm
4979874453792,ステンレス十字穴付きトラスタッピング 3×12mm,ステントラスタッピング 3×12mm,ステントラスタッピング,八幡ねじ,3×12mm
4979874220462,八幡ねじ 中空用アンカー 4S よーと,八幡ねじ 中空用アンカー 4S,中空用アンカー,八幡ねじ,4S


### パターン2. 対象を一意に特定するIDを持たない

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW new_name_list_without_jan_code AS
SELECT 
  cast(collect_list(name) as STRING) as name_list
  , ai_query(
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('提供されたリストには表記揺れしているが、実態は同じ種類の品目が重複して存在しています。品目を名寄せし、一意の品目のリストを返却してください。名寄せ結果以外の文章は返却しないでください。リスト: ', name_list),
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "name_collection",
        "schema": {
          "type": "object",
          "properties": {
            "labels": {"type": "array", "items": {"type": "string"}}
          }
        }
      }
    }'
    ) AS new_names
FROM 
  item_master
;
SELECT * FROM new_name_list_without_jan_code;

name_list,new_names
"[組ねじ 4.5×10 ［20本 (4本×5セット)］ #ネジ 八幡ねじ YAHATA DIY 通販, 八幡ねじ 組ねじ 4.5×10mm, YAHATA 組ねじ, 八幡ねじ ステントラスタッピング 3×12mm, ステントラスタッピング 3×12 1セット(14個×5パック入) 取り寄せ商品, ステンレス十字穴付きトラスタッピング 3×12mm, 八幡ねじ 中空用アンカー 4S よーと, 中空用アンカー ﾁｭｳｸｳﾖｳｱﾝｶｰ | 八幡ねじ, 中空用アンカー(4コ入) ヨート4S, S字フック ステンレス 大 6個入, レック S字フックステンレス（大） H00505 6個入, レック S字フックステンレス（大） 耐荷重9kg H00505 1パック（6個）]","{""labels"": [""組ねじ"", ""ステントラスタッピング"", ""中空用アンカー"", ""S字フックステンレス（大）""]}"


In [0]:
%sql
SELECT
  t1.name
  , t2.new_names:labels as labels
  , ai_query(
      'databricks-meta-llama-3-1-405b-instruct',
      CONCAT('提供されたテキストを', labels, 'のいずれかのラベルに分類してください。分類結果以外の文章は返却しないでください。テキスト: ', t1.name)
    ) AS new_name
FROM
  item_master t1
  LEFT OUTER JOIN new_name_list_without_jan_code t2
  ;

name,labels,new_name
組ねじ 4.5×10 ［20本 (4本×5セット)］ #ネジ 八幡ねじ YAHATA DIY 通販,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",組ねじ
八幡ねじ 組ねじ 4.5×10mm,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",組ねじ
YAHATA 組ねじ,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",組ねじ
八幡ねじ ステントラスタッピング 3×12mm,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",ステントラスタッピング
ステントラスタッピング 3×12 1セット(14個×5パック入) 取り寄せ商品,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",ステントラスタッピング
ステンレス十字穴付きトラスタッピング 3×12mm,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",ステントラスタッピング
八幡ねじ 中空用アンカー 4S よーと,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",中空用アンカー
中空用アンカー ﾁｭｳｸｳﾖｳｱﾝｶｰ | 八幡ねじ,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",中空用アンカー
中空用アンカー(4コ入) ヨート4S,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",中空用アンカー
S字フック ステンレス 大 6個入,"[""組ねじ"",""ステントラスタッピング"",""中空用アンカー"",""S字フックステンレス（大）""]",S字フックステンレス（大）


## Appendix

In [0]:
%sql
USE CATALOG skato; -- 任意のカタログ名を指定する
CREATE SCHEMA IF NOT EXISTS name_collection;  -- 任意のスキーマ名を指定する
USE SCHEMA name_collection;

-- item_masterテーブルを作成する
CREATE OR REPLACE TABLE item_master ( 
    jan_code VARCHAR(20),
    name VARCHAR(100)
) USING DELTA;

-- item_masterテーブルにサンプルの値を挿入する
INSERT INTO item_master (jan_code, name) VALUES
('4979874466846', '組ねじ 4.5×10 ［20本 (4本×5セット)］ #ネジ 八幡ねじ YAHATA DIY 通販'),
('4979874466846', '八幡ねじ 組ねじ 4.5×10mm'),
('4979874466846', 'YAHATA 組ねじ '),
('4979874453792', '八幡ねじ ステントラスタッピング 3×12mm'),
('4979874453792', 'ステントラスタッピング 3×12 1セット(14個×5パック入) 取り寄せ商品'),
('4979874453792', 'ステンレス十字穴付きトラスタッピング 3×12mm'),
('4979874220462', '八幡ねじ 中空用アンカー 4S よーと'),
('4979874220462', '中空用アンカー ﾁｭｳｸｳﾖｳｱﾝｶｰ | 八幡ねじ'),
('4979874220462', '中空用アンカー(4コ入) ヨート4S'),
('4903320162143', 'S字フック ステンレス 大 6個入'),
('4903320162143', 'レック S字フックステンレス（大） H00505 6個入'),
('4903320162143', 'レック S字フックステンレス（大） 耐荷重9kg H00505 1パック（6個）');

-- データを確認する
SELECT * FROM item_master;